Max Quan,
Zoe Frank

# Imports and EDA

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Lasso, LassoCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)

historical_data_url = 'https://raw.githubusercontent.com/MattDBailey/ANOP330/refs/heads/main/Data/BucknellLendingClubHistoricalData.csv'
current_data_url = 'https://raw.githubusercontent.com/MattDBailey/ANOP330/refs/heads/main/Data/BucknellLendingClubCurrentData.csv'

# Load historical data
df_historical = pd.read_csv(historical_data_url)
print('Historical Data:')
print('First 10 rows:')
display(df_historical.head(10))
print('\nNull values:')
display(df_historical.isnull().sum())
print('\nShape:')
print(df_historical.shape)

# Load current data
df_current = pd.read_csv(current_data_url)
print('\n\nCurrent Data:')
print('First 10 rows:')
display(df_current.head(10))
print('\nNull values:')
display(df_current.isnull().sum())
print('\nShape:')
print(df_current.shape)


Historical Data:
First 10 rows:


,id,loan_amnt,funded_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,...,fico_range_high,fico_range_low,revol_bal,revol_util,total_pymnt,last_pymnt_d,recoveries,loan_length,term_num,ret_PESS
0,44756846,10000,10000,36 months,9.9,322.63,B,10+ years,MORTGAGE,43717.0,...,794,790,59708,19.0,11485.6780,2017-06-01,0.00,26.021069,36,4.952260
1,99480680,34050,34050,60 months,28.6,1074.40,F,10+ years,MORTGAGE,85000.0,...,699,695,16779,76.0,41436.5400,2017-12-01,0.00,9.035093,60,4.338643
2,88321792,9600,9600,36 months,10.4,311.98,B,< 1 year,OWN,37000.0,...,684,680,10148,48.0,10528.5460,2017-09-01,0.00,13.010534,36,3.224118
3,58401079,5000,5000,36 months,9.9,161.32,B,3 years,MORTGAGE,120000.0,...,674,670,14607,38.0,5480.6100,2016-10-01,0.00,13.010534,36,3.204066
4,92749250,9000,9000,36 months,17.9,325.33,D,< 1 year,RENT,70000.0,...,664,660,5344,66.0,10473.4795,2017-11-01,0.00,11.992033,36,5.457331
5,18495553,12000,12000,60 months,20.9,324.58,E,8 years,RENT,38000.0,...,674,670,3884,73.0,2604.7500,2015-03-01,0.00,8.969383,60,-15.658751
6,31507182,4800,4800,36 months,11.6,158.68,B,2 years,RENT,75000.0,...,719,715,12788,56.0,1739.2600,2015-09-01,0.00,11.006386,36,-21.255138
7,97924159,14625,14625,60 months,28.6,461.47,F,3 years,OWN,43920.0,...,719,715,12154,31.0,20615.1290,2018-09-01,0.00,18.957268,60,8.191629
8,774919,14000,14000,36 months,16.4,495.60,D,4 years,RENT,45000.0,...,684,680,3898,40.0,4687.9900,2011-12-01,1720.93,6.012444,36,-22.171452
9,93207299,20000,20000,36 months,11.4,658.95,B,4 years,RENT,80000.0,...,664,660,17339,41.0,22647.7150,2018-04-01,0.00,16.953120,36,4.412858



Null values:


,0
id,0
loan_amnt,0
funded_amnt,0
term,0
int_rate,0
...,...
last_pymnt_d,0
recoveries,0
loan_length,0
term_num,0



Shape:
(50000, 29)


Current Data:
First 10 rows:


,id,loan_amnt,term,int_rate,installment,grade,emp_length,home_ownership,annual_inc,verification_status,purpose,dti,delinq_2yrs,earliest_cr_line,open_acc,pub_rec,fico_range_high,fico_range_low,revol_bal,revol_util
0,54097739,9800,36 months,7.8,306.60,A,7 years,OWN,50000.0,Source Verified,debt_consolidation,33.73,0.0,1998-09-01,11.0,0.0,744,740,8069,31.0
1,38479328,6000,36 months,7.4,186.61,A,5 years,RENT,65000.0,Source Verified,credit_card,10.08,0.0,1994-11-01,9.0,1.0,699,695,4867,33.0
2,85481549,35000,60 months,24.4,1016.86,E,10+ years,RENT,70000.0,Source Verified,other,10.68,1.0,1990-03-01,6.0,0.0,679,675,14993,69.0
3,33181666,12000,36 months,8.6,379.76,B,1 year,RENT,63000.0,Source Verified,debt_consolidation,22.99,0.0,2003-09-01,17.0,0.0,674,670,10751,69.0
4,52016730,16000,36 months,13.3,541.65,C,8 years,MORTGAGE,90000.0,Not Verified,debt_consolidation,4.49,0.0,2000-07-01,13.0,1.0,689,685,10958,37.0
5,72235609,18000,36 months,7.8,563.15,A,10+ years,MORTGAGE,120000.0,Source Verified,debt_consolidation,25.48,0.0,1998-12-01,29.0,0.0,694,690,24962,31.0
6,22111464,14000,36 months,7.6,436.71,A,4 years,RENT,100000.0,Not Verified,debt_consolidation,19.91,1.0,1999-01-01,9.0,0.0,714,710,39787,68.0
7,137616497,10000,36 months,6.1,304.72,A,3 years,RENT,100000.0,Not Verified,debt_consolidation,8.93,0.0,2001-11-01,8.0,0.0,779,775,9343,19.0
8,56925226,6375,36 months,6.8,196.53,A,4 years,MORTGAGE,50460.0,Not Verified,credit_card,13.86,0.0,2008-05-01,7.0,0.0,724,720,6496,42.0
9,43639723,12000,36 months,11.5,395.89,B,1 year,RENT,33000.0,Not Verified,debt_consolidation,21.42,0.0,2003-08-01,8.0,0.0,689,685,10755,66.0



Null values:


,0
id,0
loan_amnt,0
term,0
int_rate,0
installment,0
grade,0
emp_length,66
home_ownership,0
annual_inc,0
verification_status,0



Shape:
(1000, 20)


In [3]:
columns_to_drop = ['total_pymnt', 'recoveries', 'last_pymnt_d', 'loan_length', 'id']

# Drop columns from historical data
df_historical = df_historical.drop(columns=columns_to_drop, errors='ignore')
print('Historical Data after dropping columns:')
print(df_historical.shape)

# Identify columns to drop that exist in df_current
existing_columns_in_current = [col for col in columns_to_drop if col in df_current.columns]

# Drop only existing columns from current data
df_current = df_current.drop(columns=existing_columns_in_current, errors='ignore')
print('\nCurrent Data after dropping columns:')
print(df_current.shape)

Historical Data after dropping columns:
(50000, 24)

Current Data after dropping columns:
(1000, 19)


In [4]:
import pandas as pd
import numpy as np

# Create copies of the DataFrames to work with, to avoid modifying the original dataframes directly
df_historical_temp = df_historical.copy()
df_current_temp = df_current.copy()

# Identify categorical columns to encode (excluding date-like columns like 'issue_d' and 'earliest_cr_line')
# These are columns that are 'object' dtype based on prior exploration
categorical_cols_to_encode = [
    'term', 'grade', 'emp_length', 'home_ownership', 'verification_status', 'purpose'
]

# Identify categorical columns specific to historical data (e.g., 'loan_status' might be a feature)
categorical_cols_hist_only = ['loan_status']

# --- Step 1: Collect all unique categories for each column across both dataframes ---
# This ensures consistency in dummy variable creation, even if a category is only present in one df
all_categories = {}
for col in categorical_cols_to_encode:
    unique_vals_hist = df_historical_temp[col].dropna().unique() if col in df_historical_temp.columns else np.array([])
    unique_vals_curr = df_current_temp[col].dropna().unique() if col in df_current_temp.columns else np.array([])
    all_categories[col] = np.unique(np.concatenate((unique_vals_hist, unique_vals_curr))).tolist()

# Add categories for columns only in historical data if they are meant to be encoded
for col in categorical_cols_hist_only:
    if col in df_historical_temp.columns:
        all_categories[col] = df_historical_temp[col].dropna().unique().tolist()


# --- Step 2: Convert columns to Pandas Categorical type with explicit categories ---
# This step is crucial for `pd.get_dummies` to generate consistent columns across both DataFrames
for col in categorical_cols_to_encode:
    if col in df_historical_temp.columns:
        df_historical_temp[col] = pd.Categorical(df_historical_temp[col], categories=all_categories[col])
    if col in df_current_temp.columns:
        df_current_temp[col] = pd.Categorical(df_current_temp[col], categories=all_categories[col])

# For historical-only categorical columns
for col in categorical_cols_hist_only:
    if col in df_historical_temp.columns:
        df_historical_temp[col] = pd.Categorical(df_historical_temp[col], categories=all_categories[col])


# --- Step 3: Apply get_dummies to both dataframes ---
# `drop_first=True` is used to avoid multicollinearity
# `dummy_na=False` (default) means NaN values will not create a separate dummy column
df_historical_encoded = pd.get_dummies(df_historical_temp,
                                        columns=categorical_cols_to_encode + categorical_cols_hist_only,
                                        drop_first=True)

df_current_encoded = pd.get_dummies(df_current_temp,
                                      columns=categorical_cols_to_encode,
                                      drop_first=True)


# --- Step 4: Align columns between historical and current dataframes ---
# Assuming 'ret_PESS' is the target variable in df_historical_encoded and should not be in df_current_encoded
target_column = 'ret_PESS'

# Get the list of feature columns from the historical encoded dataframe,
# excluding the target column as current data will not have it (it's what we predict)
historical_feature_cols = [col for col in df_historical_encoded.columns if col != target_column]

# Reindex df_current_encoded to match the feature columns of historical data.
# This will add any missing feature columns (filled with 0) and drop any extra ones from current.
df_current_aligned = df_current_encoded.reindex(columns=historical_feature_cols, fill_value=0)

# Ensure the order of columns in df_current_aligned is the same as in historical_feature_cols
df_current_aligned = df_current_aligned[historical_feature_cols]


# --- Step 5: Update the original dataframes with the encoded versions ---
df_historical = df_historical_encoded
df_current = df_current_aligned

# Display results to show the new shapes and a preview of the data
print('Shape of df_historical after encoding:', df_historical.shape)
print('Shape of df_current after encoding and alignment:', df_current.shape)

print('\nFirst 5 rows of df_historical (encoded):')
display(df_historical.head())
print('\nFirst 5 rows of df_current (encoded and aligned):')
display(df_current.head())

Shape of df_historical after encoding: (50000, 55)
Shape of df_current after encoding and alignment: (1000, 54)

First 5 rows of df_historical (encoded):


,loan_amnt,funded_amnt,int_rate,installment,annual_inc,issue_d,dti,delinq_2yrs,earliest_cr_line,open_acc,...,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,loan_status_Charged Off
0,10000,10000,9.9,322.63,43717.0,2015-04-01,6.20,0.0,1997-03-01,7.0,...,False,False,False,False,False,False,False,False,False,False
1,34050,34050,28.6,1074.40,85000.0,2017-03-01,16.91,1.0,2004-09-01,11.0,...,False,True,False,False,False,False,False,False,False,False
2,9600,9600,10.4,311.98,37000.0,2016-08-01,24.29,0.0,1984-05-01,14.0,...,False,False,False,False,False,False,False,False,False,False
3,5000,5000,9.9,161.32,120000.0,2015-09-01,13.03,0.0,2008-07-01,12.0,...,False,False,False,False,False,False,False,False,False,False
4,9000,9000,17.9,325.33,70000.0,2016-11-01,11.18,0.0,2012-09-01,6.0,...,False,False,False,False,False,False,False,False,False,False



First 5 rows of df_current (encoded and aligned):


,loan_amnt,funded_amnt,int_rate,installment,annual_inc,issue_d,dti,delinq_2yrs,earliest_cr_line,open_acc,...,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,loan_status_Charged Off
0,9800,0,7.8,306.60,50000.0,0,33.73,0.0,1998-09-01,11.0,...,False,False,False,False,False,False,False,False,False,0
1,6000,0,7.4,186.61,65000.0,0,10.08,0.0,1994-11-01,9.0,...,False,False,False,False,False,False,False,False,False,0
2,35000,0,24.4,1016.86,70000.0,0,10.68,1.0,1990-03-01,6.0,...,False,False,False,False,True,False,False,False,False,0
3,12000,0,8.6,379.76,63000.0,0,22.99,0.0,2003-09-01,17.0,...,False,False,False,False,False,False,False,False,False,0
4,16000,0,13.3,541.65,90000.0,0,4.49,0.0,2000-07-01,13.0,...,False,False,False,False,False,False,False,False,False,0


In [5]:
print('Number of unique values per column in Historical Data (df_historical):')
for col in df_historical.columns:
    print(f"{col}: {df_historical[col].nunique()}")

print('\nNumber of unique values per column in Current Data (df_current):')
for col in df_current.columns:
    print(f"{col}: {df_current[col].nunique()}")

Number of unique values per column in Historical Data (df_historical):
loan_amnt: 1298
funded_amnt: 1299
int_rate: 220
installment: 19095
annual_inc: 5063
issue_d: 120
dti: 4137
delinq_2yrs: 18
earliest_cr_line: 619
open_acc: 52
pub_rec: 14
fico_range_high: 38
fico_range_low: 38
revol_bal: 26643
revol_util: 127
term_num: 2
ret_PESS: 47950
term_ 60 months: 2
grade_B: 2
grade_C: 2
grade_D: 2
grade_E: 2
grade_F: 2
grade_G: 2
emp_length_10+ years: 2
emp_length_2 years: 2
emp_length_3 years: 2
emp_length_4 years: 2
emp_length_5 years: 2
emp_length_6 years: 2
emp_length_7 years: 2
emp_length_8 years: 2
emp_length_9 years: 2
emp_length_< 1 year: 2
home_ownership_MORTGAGE: 2
home_ownership_NONE: 2
home_ownership_OTHER: 2
home_ownership_OWN: 2
home_ownership_RENT: 2
verification_status_Source Verified: 2
verification_status_Verified: 2
purpose_credit_card: 2
purpose_debt_consolidation: 2
purpose_educational: 2
purpose_home_improvement: 2
purpose_house: 2
purpose_major_purchase: 2
purpose_medica

# Training the Data

In [6]:
# Define features (X) and target (y)
# Ensure to use 'term_ 60 months' as 'term' was encoded
features = ['int_rate', 'fico_range_high', 'loan_amnt', 'term_num', 'installment']
target = 'ret_PESS'

X = df_historical[features]
y = df_historical[target]

# Split historical data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (40000, 5)
X_test shape: (10000, 5)
y_train shape: (40000,)
y_test shape: (10000,)


In [7]:
# Initialize StandardScaler
scaler = StandardScaler()

# Fit on training data and transform both training and test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame to maintain column names and index for clarity
X_train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("Data scaled successfully.")
print(f"X_train (scaled) shape: {X_train.shape}")
print(f"X_test (scaled) shape: {X_test.shape}")

Data scaled successfully.
X_train (scaled) shape: (40000, 5)
X_test (scaled) shape: (10000, 5)


# Linear Regression

In [8]:
# 1. Perform a basic Linear Regression
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = linear_model.predict(X_test)

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mape = mean_absolute_percentage_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")
print(f"R-squared (R2): {r2:.2f}")

Mean Absolute Error (MAE): 5.63
Mean Absolute Percentage Error (MAPE): 1.11
Mean Squared Error (MSE): 67.93
Root Mean Squared Error (RMSE): 8.24
R-squared (R2): 0.01


# Lasso CV

In [9]:
# Perform Lasso CV
lasso_cv_model = LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=10000)
lasso_cv_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred_lasso = lasso_cv_model.predict(X_test)

# Calculate evaluation metrics for Lasso CV
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mape_lasso = mean_absolute_percentage_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mse_lasso)
r2_lasso = r2_score(y_test, y_pred_lasso)

print(f"LassoCV - Best alpha: {lasso_cv_model.alpha_:.4f}")
print(f"LassoCV - Mean Absolute Error (MAE): {mae_lasso:.2f}")
print(f"LassoCV - Mean Absolute Percentage Error (MAPE): {mape_lasso:.2f}")
print(f"LassoCV - Mean Squared Error (MSE): {mse_lasso:.2f}")
print(f"LassoCV - Root Mean Squared Error (RMSE): {rmse_lasso:.2f}")
print(f"LassoCV - R-squared (R2): {r2_lasso:.2f}")

LassoCV - Best alpha: 0.0007
LassoCV - Mean Absolute Error (MAE): 5.63
LassoCV - Mean Absolute Percentage Error (MAPE): 1.11
LassoCV - Mean Squared Error (MSE): 67.93
LassoCV - Root Mean Squared Error (RMSE): 8.24
LassoCV - R-squared (R2): 0.01


# Compare the Models

In [10]:
results_data = {
    'Metric': ['MAE', 'MAPE', 'MSE', 'RMSE', 'R-squared'],
    'Linear Regression': [mae, mape, mse, rmse, r2],
    'LassoCV': [mae_lasso, mape_lasso, mse_lasso, rmse_lasso, r2_lasso]
}

df_results = pd.DataFrame(results_data)
display(df_results)

,Metric,Linear Regression,LassoCV
0,MAE,5.625330,5.625407
1,MAPE,1.107228,1.106284
2,MSE,67.928771,67.927105
3,RMSE,8.241891,8.241790
4,R-squared,0.011919,0.011943


In [11]:
df_results.to_pickle('churn_model.pkl')
print("DataFrame 'df_results' saved as 'churn_model.pkl'")

DataFrame 'df_results' saved as 'churn_model.pkl'
